# Urdu Aegis voice — LoRA fine-tune for English loanwords

**One self-contained notebook.** Upload it + your `dataset.zip` to Colab,
set **Runtime → Change runtime type → GPU (T4)**, then **Run all**.

Teaches `ur_PK-aegis_female-medium` to say *keypad / card / transaction / OTP*
inside Urdu by training tiny low-rank adapters on the **frozen** model
(~1% of weights), then folding them into the weights → a normal Piper ONNX.
Every base weight stays byte-identical, so plain Urdu can't regress.

## You need a base checkpoint

Piper needs a trainable `.ckpt`. Aegis only ships an ONNX, so put the real
**Aegis student checkpoint** at `/content/aegis-female-init.ckpt` before
running (Files pane, or `hf_hub_download`). Ask Mahwiz for it —
`request_to_mahwiz.md`. If it is absent the notebook warm-starts from the
public **Fasih** ckpt so the pipeline still runs, but Fasih is the **male**
Urdu voice, so the result is male until you supply the real ckpt.
**Cell 5b plays the base voice before training — check it there.**

**`dataset.zip`** must contain `metadata.csv` (lines `wav/0001.wav|<urdu text>`,
`|`-delimited) and a `wav/` folder of 22 050 Hz mono WAVs. Cell 3 either
unzips an uploaded one, or (fallback) rebuilds it from Uplift AI if you
paste a key.

Runtime ~30-45 min on a T4. Outputs: `ur_PK-aegis_female-medium.onnx` + `.json`.

## 1 · GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> GPU'
print(torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 2 · Install piper1-gpl (training) — ~6-8 min

`scikit-build` etc. must be present **before** the build (it compiles `piper.espeakbridge`, which the training phonemiser needs).

In [ ]:
%%bash
set -e
apt-get -qq update >/dev/null 2>&1 || true
apt-get -qq install -y build-essential cmake ninja-build espeak-ng >/dev/null 2>&1 || true
pip -q install scikit-build cmake ninja 'cython>=3,<4'
[ -d /content/piper1-gpl ] || git clone -q https://github.com/OHF-voice/piper1-gpl.git /content/piper1-gpl
pip -q install -e '/content/piper1-gpl[train]'
cd /content/piper1-gpl
bash ./build_monotonic_align.sh
python setup.py -q build_ext --inplace
# new torch ONNX exporter (dynamo) cannot trace piper's stochastic duration
# predictor -> force the legacy TorchScript exporter
sed -i 's/torch.onnx.export(/torch.onnx.export(dynamo=False,/' /content/piper1-gpl/src/piper/train/export_onnx.py
pip -q install onnx onnxscript huggingface_hub soundfile
echo "--- import check ---"
python -c "import piper.espeakbridge; from piper.train.vits.monotonic_align.core import maximum_path_c; from piper.train.vits.lightning import VitsModel; from piper.train.vits.dataset import VitsDataModule; print('training imports OK')"


## 3 · Dataset — upload `dataset.zip` (or rebuild from Uplift AI)

In [ ]:
# leave blank to use an uploaded dataset.zip; paste a key to (re)build it
UPLIFT_API_KEY = ""          # sk_api_...  (optional fallback)
UPLIFT_VOICE   = "helpdesk-agent"

In [ ]:
SENTENCES = [
"حفاظت کے لیے، براہِ کرم اپنے شناختی کارڈ کے آخری چھ ہندسے اب اپنے کی پیڈ پر درج کریں۔",
"براہِ کرم اپنے کارڈ کے آخری چار ہندسے اب اپنے کی پیڈ پر درج کریں، انہیں بول کر نہ بتائیں۔",
"براہِ کرم اپنی تاریخِ پیدائش سال، مہینہ، دن کی ترتیب میں اپنے کی پیڈ پر درج کریں۔",
"میرے کارڈ پر ایک فراڈ ٹرانزیکشن ہے، براہِ کرم اسے بلاک کر دیں۔",
"میرے کریڈٹ کارڈ پر کتنی رقم واجب الادا ہے؟",
"مجھے نئی چیک بک چاہیے، براہِ کرم میرے رجسٹرڈ پتے پر بھیج دیں۔",
"میرا کارڈ گم ہو گیا ہے، اسے فوراً بلاک کر دیں۔",
"میں آپ سے آپ کا پن، سی وی وی، او ٹی پی یا پاس ورڈ کبھی نہیں پوچھوں گی۔",
"آپ کا کارڈ بلاک کر دیا گیا ہے اور آپ کے موبائل نمبر پر ایس ایم ایس بھیج دیا گیا ہے۔",
"آپ کے اکاؤنٹ کا بیلنس پچاس ہزار روپے ہے۔",
"آپ کے کریڈٹ کارڈ کی لمٹ پانچ لاکھ روپے ہے اور دستیاب رقم چودہ ہزار روپے ہے۔",
"آپ کی کمپلینٹ رجسٹرڈ ہو گئی ہے، ریفرنس نمبر نوٹ کر لیں۔",
"آپ کی ٹرانسفر کی ریکوئسٹ پراسیس ہو رہی ہے۔",
"براہِ کرم ایچ بی ایل موبائل ایپ کھولیں اور آن لائن اسٹیٹمنٹ دیکھیں۔",
"آپ کا اے ٹی ایم کارڈ ایکسپائر ہو چکا ہے، نیا کارڈ برانچ سے وصول کریں۔",
"آپ کے کارڈ کی تفصیل اسکرین پر موجود ہے۔",
"آپ کے اکاؤنٹ کی تفصیل اسکرین پر موجود ہے۔",
"آپ کے بیلنس کی تفصیل اسکرین پر موجود ہے۔",
"آپ کے اسٹیٹمنٹ کی تفصیل اسکرین پر موجود ہے۔",
"آپ کے ٹرانزیکشن کی تفصیل اسکرین پر موجود ہے۔",
"آپ کے لمٹ کی تفصیل اسکرین پر موجود ہے۔",
"براہِ کرم اپنا پن کسی کے ساتھ شیئر نہ کریں۔",
"براہِ کرم اپنا او ٹی پی کسی کے ساتھ شیئر نہ کریں۔",
"براہِ کرم اپنا سی وی وی کسی کے ساتھ شیئر نہ کریں۔",
"براہِ کرم اپنا پاس ورڈ کسی کے ساتھ شیئر نہ کریں۔",
"براہِ کرم اپنا کوڈ کسی کے ساتھ شیئر نہ کریں۔",
"میں آپ کے اکاؤنٹ کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"میں آپ کے کارڈ کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"میں آپ کے ٹرانزیکشن کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"میں آپ کے بیلنس کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"میں آپ کے سٹیٹس کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"میں آپ کے اسٹیٹمنٹ کی جانچ کر رہی ہوں، براہِ کرم انتظار کریں۔",
"آپ کا موبائل کامیابی سے اپڈیٹ ہو گیا ہے۔",
"آپ کا ای میل کامیابی سے اپڈیٹ ہو گیا ہے۔",
"آپ کا پن کامیابی سے اپڈیٹ ہو گیا ہے۔",
"آپ کا پاس ورڈ کامیابی سے اپڈیٹ ہو گیا ہے۔",
"آپ کے کارڈ پر کوئی مشکوک سرگرمی نظر نہیں آئی۔",
"آپ کے اکاؤنٹ پر کوئی مشکوک سرگرمی نظر نہیں آئی۔",
"آپ کے والیٹ پر کوئی مشکوک سرگرمی نظر نہیں آئی۔",
"آپ کا کارڈ بلاک کر دیا گیا ہے اور نیا جاری کر دیا گیا ہے۔",
"آپ کا اے ٹی ایم بلاک کر دیا گیا ہے اور نیا جاری کر دیا گیا ہے۔",
"آپ ایپ کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ آن لائن کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ انٹرنیٹ کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ ایس ایم ایس کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ اے ٹی ایم کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ کال کے ذریعے بھی یہ کام کر سکتے ہیں۔",
"آپ کی کمپلینٹ موصول ہو گئی ہے اور اس پر کارروائی جاری ہے۔",
"آپ کا ٹرانزیکشن چارج پانچ سو روپے ہے۔",
"آپ کا سروس چارج پانچ سو روپے ہے۔",
"آپ کا کارڈ چارج پانچ سو روپے ہے۔",
"براہِ کرم اپنا کی پیڈ دوبارہ چیک کریں۔",
"آپ کا کی پیڈ تیار ہے۔",
"یہ کی پیڈ محفوظ رکھیں۔",
"میں آپ کو کی پیڈ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا کارڈ دوبارہ چیک کریں۔",
"آپ کا کارڈ تیار ہے۔",
"یہ کارڈ محفوظ رکھیں۔",
"میں آپ کو کارڈ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا او ٹی پی دوبارہ چیک کریں۔",
"آپ کا او ٹی پی تیار ہے۔",
"یہ او ٹی پی محفوظ رکھیں۔",
"میں آپ کو او ٹی پی کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا سی این آئی سی دوبارہ چیک کریں۔",
"آپ کا سی این آئی سی تیار ہے۔",
"یہ سی این آئی سی محفوظ رکھیں۔",
"میں آپ کو سی این آئی سی کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ایچ بی ایل دوبارہ چیک کریں۔",
"آپ کا ایچ بی ایل تیار ہے۔",
"یہ ایچ بی ایل محفوظ رکھیں۔",
"میں آپ کو ایچ بی ایل کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا اے ٹی ایم دوبارہ چیک کریں۔",
"آپ کا اے ٹی ایم تیار ہے۔",
"یہ اے ٹی ایم محفوظ رکھیں۔",
"میں آپ کو اے ٹی ایم کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ایس ایم ایس دوبارہ چیک کریں۔",
"آپ کا ایس ایم ایس تیار ہے۔",
"یہ ایس ایم ایس محفوظ رکھیں۔",
"میں آپ کو ایس ایم ایس کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا پن دوبارہ چیک کریں۔",
"آپ کا پن تیار ہے۔",
"یہ پن محفوظ رکھیں۔",
"میں آپ کو پن کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا سی وی وی دوبارہ چیک کریں۔",
"آپ کا سی وی وی تیار ہے۔",
"یہ سی وی وی محفوظ رکھیں۔",
"میں آپ کو سی وی وی کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ٹرانزیکشن دوبارہ چیک کریں۔",
"آپ کا ٹرانزیکشن تیار ہے۔",
"یہ ٹرانزیکشن محفوظ رکھیں۔",
"میں آپ کو ٹرانزیکشن کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا بیلنس دوبارہ چیک کریں۔",
"آپ کا بیلنس تیار ہے۔",
"یہ بیلنس محفوظ رکھیں۔",
"میں آپ کو بیلنس کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا اسٹیٹمنٹ دوبارہ چیک کریں۔",
"آپ کا اسٹیٹمنٹ تیار ہے۔",
"یہ اسٹیٹمنٹ محفوظ رکھیں۔",
"میں آپ کو اسٹیٹمنٹ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا پاس ورڈ دوبارہ چیک کریں۔",
"آپ کا پاس ورڈ تیار ہے۔",
"یہ پاس ورڈ محفوظ رکھیں۔",
"میں آپ کو پاس ورڈ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ایپ دوبارہ چیک کریں۔",
"آپ کا ایپ تیار ہے۔",
"یہ ایپ محفوظ رکھیں۔",
"میں آپ کو ایپ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا کریڈٹ کارڈ دوبارہ چیک کریں۔",
"آپ کا کریڈٹ کارڈ تیار ہے۔",
"یہ کریڈٹ کارڈ محفوظ رکھیں۔",
"میں آپ کو کریڈٹ کارڈ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ڈیبٹ کارڈ دوبارہ چیک کریں۔",
"آپ کا ڈیبٹ کارڈ تیار ہے۔",
"یہ ڈیبٹ کارڈ محفوظ رکھیں۔",
"میں آپ کو ڈیبٹ کارڈ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا برانچ دوبارہ چیک کریں۔",
"آپ کا برانچ تیار ہے۔",
"یہ برانچ محفوظ رکھیں۔",
"میں آپ کو برانچ کے بارے میں بتاتی ہوں۔",
"براہِ کرم اپنا ٹرانسفر دوبارہ چیک کریں۔",
"آپ کا ٹرانسفر تیار ہے۔",
"یہ ٹرانسفر محفوظ رکھیں۔",
"میں آپ کو ٹرانسفر کے بارے میں بتاتی ہوں۔"
]

In [ ]:
import pathlib, zipfile, shutil, wave

CANON = pathlib.Path("/content/dataset")          # canonical layout: metadata.csv + wav/
(CANON / "wav").mkdir(parents=True, exist_ok=True)

def _have_dataset():
    m = CANON / "metadata.csv"
    if not m.is_file():
        return False
    n = len([x for x in m.read_text(encoding="utf-8").splitlines() if x.strip()])
    return n > 0 and len(list((CANON / "wav").glob("*.wav"))) >= n

if not _have_dataset():
    # 1. get a zip
    z = pathlib.Path("/content/dataset.zip")
    if not z.is_file():
        try:
            from google.colab import files
            print("Upload dataset.zip (metadata.csv + wav/ inside)…")
            z = pathlib.Path("/content") / list(files.upload())[0]
        except Exception:
            z = None
    if z and z.is_file():
        tmp = pathlib.Path("/content/_dsraw"); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
        with zipfile.ZipFile(z) as zf:
            zf.extractall(tmp)
        meta = next(iter(tmp.rglob("metadata.csv")), None)
        assert meta is not None, "no metadata.csv in the zip"
        src_root = meta.parent
        rows = []
        for line in meta.read_text(encoding="utf-8").splitlines():
            if "|" not in line:
                continue
            rel, text = line.split("|", 1)
            cand = (src_root / rel)
            if not cand.is_file():
                cand = next(iter(src_root.rglob(pathlib.Path(rel).name)), None)
            assert cand and cand.is_file(), f"missing wav for {rel}"
            name = pathlib.Path(rel).name
            shutil.copy(cand, CANON / "wav" / name)
            rows.append(f"wav/{name}|{text.strip()}")
        (CANON / "metadata.csv").write_text("\n".join(rows) + "\n", encoding="utf-8")
        shutil.rmtree(tmp, ignore_errors=True)

# 2. Uplift fallback
if not _have_dataset() and UPLIFT_API_KEY.startswith("sk_"):
    import requests
    rows = []
    for i, t in enumerate(SENTENCES, 1):
        r = requests.post("https://api.upliftai.org/v1/synthesis/text-to-speech",
            headers={"Authorization": f"Bearer {UPLIFT_API_KEY}"},
            json={"voiceId": UPLIFT_VOICE, "text": t, "outputFormat": "WAV_22050_16"}, timeout=90)
        r.raise_for_status()
        (CANON / "wav" / f"{i:04d}.wav").write_bytes(r.content)
        rows.append(f"wav/{i:04d}.wav|{t}")
        if i % 20 == 0: print(f"  {i}/{len(SENTENCES)}")
    (CANON / "metadata.csv").write_text("\n".join(rows) + "\n", encoding="utf-8")

assert _have_dataset(), "no dataset — upload dataset.zip or paste an Uplift key in the cell above"
DATASET_DIR = str(CANON)
_rows = [l for l in (CANON / "metadata.csv").read_text(encoding="utf-8").splitlines() if l.strip()]
_w0 = sorted((CANON / "wav").glob("*.wav"))[0]
assert wave.open(str(_w0)).getframerate() == 22050, "WAVs must be 22050 Hz"
_d = sum(wave.open(str(w)).getnframes() / 22050 for w in (CANON / "wav").glob("*.wav"))
print(f"dataset OK: {len(_rows)} clips at {DATASET_DIR}  ({_d/60:.1f} min)")
print("  first line:", _rows[0][:70])


## 4 · Base models (public — Fasih ckpt + Aegis ONNX)

In [ ]:
from huggingface_hub import hf_hub_download
import shutil
shutil.copy(hf_hub_download("rhasspy/piper-checkpoints",
    "ur/ur_PK/fasih/medium/epoch=3206-step=383452.ckpt", repo_type="dataset"),
    "/content/fasih.ckpt")
shutil.copy(hf_hub_download("mahwizzzz/piper-voice-ur-aegis-female",
    "ur_PK-aegis_female-medium.onnx"), "/content/aegis.onnx")
shutil.copy(hf_hub_download("mahwizzzz/piper-voice-ur-aegis-female",
    "ur_PK-aegis_female-medium.onnx.json"), "/content/aegis.onnx.json")
print("ok")


## 5 · Trainable base checkpoint

Put the real **Aegis student checkpoint** at `/content/aegis-female-init.ckpt`
(Files pane, or `hf_hub_download`). Ask Mahwiz for it — see
`finetune/request_to_mahwiz.md`. This cell only validates it.

**If it is missing**, the notebook warm-starts from **Fasih** so the pipeline
still runs end to end — but Fasih is the **male** Urdu voice, so the trained
voice will be male until you supply the real ckpt.

In [ ]:
import os, sys, json, shutil, warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, "/content/piper1-gpl/src")
import torch

CFG  = json.load(open("/content/aegis.onnx.json"))     # used later for num_symbols / phoneme_id_map
INIT = "/content/aegis-female-init.ckpt"

if os.path.exists(INIT):
    _sd = torch.load(INIT, map_location="cpu")
    _st = _sd.get("state_dict", _sd)
    assert any(k.startswith("model_g.") for k in _st), \
        f"{INIT} is not a piper Lightning checkpoint (no model_g.* keys)"
    print(f"OK  {INIT}  ({os.path.getsize(INIT)/1e6:.0f} MB, {len(_st)} tensors)")
    print("    -> real Aegis-female base. Run cell 5b to confirm it sounds right.")
    del _sd, _st
else:
    print(f"!!  {INIT} not found.")
    print("!!  Falling back to the FASIH warm-start (public).")
    print("!!  Fasih is the MALE Urdu voice - the trained voice will be MALE")
    print("!!  until you drop the real Aegis student ckpt at that path.")
    print("!!  How to get it: finetune/request_to_mahwiz.md")
    shutil.copy("/content/fasih.ckpt", INIT)
    print(f"    wrote {INIT}  (= Fasih male)")

## 5b · Test the base checkpoint *before* training

Synthesise a few lines straight from `/content/aegis-female-init.ckpt` with no
adapter. This must be **clean, natural speech** — and, with the real Aegis
ckpt, **female**. If it is noise, the checkpoint is wrong: stop here, don't
train on it.

In [ ]:
import numpy as np, gc, torch
from piper import PiperVoice
from piper.train.vits.lightning import VitsModel
from IPython.display import Audio, display, Markdown

_pid = CFG["phoneme_id_map"]
_pv  = PiperVoice.load("/content/aegis.onnx")            # phonemisation only

_probe = VitsModel(num_symbols=CFG["num_symbols"], num_speakers=1,
                   sample_rate=22050, batch_size=8, mos_metric="none")
_m, _u = _probe.load_state_dict(
    torch.load("/content/aegis-female-init.ckpt", map_location="cpu")["state_dict"],
    strict=False)
print(f"loaded base: {len(_m)} missing, {len(_u)} unexpected")
_probe.model_g.eval()

for _t in ["آج میں آپ کی کیا مدد کر سکتی ہوں؟",
           "اپنے فون کے کی پیڈ پر چار ہندسے دبائیں۔",
           "آپ کے کارڈ پر ایک فراڈ ٹرانزیکشن ہے۔"]:
    _ids = [1]
    for _p in (x for s in _pv.phonemize(_t) for x in s):
        if _p in _pid: _ids += _pid[_p] + [0]
    _ids += [2]
    with torch.no_grad():
        _a = _probe.model_g.infer(torch.LongTensor([_ids]), torch.LongTensor([len(_ids)]),
                noise_scale=0.667, length_scale=1.0, noise_scale_w=0.8)[0][0, 0].numpy()
    display(Markdown(f"`{_t}`")); display(Audio(_a, rate=22050))

del _probe, _pv; gc.collect()
try: torch.cuda.empty_cache()
except Exception: pass
print("clean speech -> continue.  noise -> the ckpt is bad, do not train.")

## 6 · The low-rank adapter

In [ ]:
# Low-rank adapters for a piper1-gpl VITS model.
# Train ONLY these; every base weight stays frozen and byte-identical, so
# plain-Urdu output cannot regress. At export they fold into the weights ->
# a normal Piper ONNX. set_lora_scale(0) == the exact original voice.
import torch
import torch.nn as nn


class LoRAConv1d(nn.Module):
    """Frozen base Conv1d + trainable low-rank branch (up-proj zero-init)."""

    def __init__(self, base: nn.Conv1d, rank: int = 8, alpha: float = 8.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.rank = rank
        self.base_scaling = alpha / rank
        self.scaling = self.base_scaling
        self.enabled = True
        k = base.kernel_size[0]
        pad = base.padding[0] if isinstance(base.padding, tuple) else base.padding
        self.lora_down = nn.Conv1d(base.in_channels, rank, k, stride=base.stride,
                                   padding=pad, dilation=base.dilation, bias=False)
        self.lora_up = nn.Conv1d(rank, base.out_channels, 1, bias=False)
        nn.init.kaiming_uniform_(self.lora_down.weight, a=5 ** 0.5)
        nn.init.zeros_(self.lora_up.weight)

    def forward(self, x):
        out = self.base(x)
        if self.enabled:
            out = out + self.lora_up(self.lora_down(x)) * self.scaling
        return out

    @torch.no_grad()
    def merged_weight(self):
        up = self.lora_up.weight.squeeze(-1)
        delta = torch.einsum("or,rik->oik", up, self.lora_down.weight)
        return self.base.weight + delta * self.scaling


def _iter_conv1d(model):
    for name, mod in model.named_modules():
        for cn, ch in list(mod.named_children()):
            if isinstance(ch, nn.Conv1d):
                yield mod, cn, f"{name}.{cn}".lstrip(".")


def _iter_lora(model):
    for name, mod in model.named_modules():
        for cn, ch in list(mod.named_children()):
            if isinstance(ch, LoRAConv1d):
                yield mod, cn


def inject_lora(model, targets, rank=8, alpha=8.0):
    n = 0
    for parent, cn, dotted in list(_iter_conv1d(model)):
        if any(t in dotted for t in targets):
            setattr(parent, cn, LoRAConv1d(getattr(parent, cn), rank, alpha))
            n += 1
    return n


def freeze_base(model):
    for name, p in model.named_parameters():
        p.requires_grad_(".lora_down." in name or ".lora_up." in name)


def lora_parameters(model):
    return [p for n, p in model.named_parameters()
            if ".lora_down." in n or ".lora_up." in n]


def set_lora_scale(model, scale):
    for m in model.modules():
        if isinstance(m, LoRAConv1d):
            m.enabled = scale != 0.0
            m.scaling = scale * m.base_scaling


@torch.no_grad()
def merge_lora(model):
    for parent, cn in list(_iter_lora(model)):
        lora = getattr(parent, cn)
        conv = lora.base
        conv.weight.data.copy_(lora.merged_weight())
        setattr(parent, cn, conv)
    return model


## 7 · Train (base frozen, only the adapter learns)

In [ ]:
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, TQDMProgressBar
from piper.train.vits.lightning import VitsModel
from piper.train.vits.dataset import VitsDataModule

RANK, ALPHA, LR, MAX_STEPS, BATCH = 8, 8, 2e-4, 1500, 8
TARGETS = ["enc_p.encoder.attn_layers", "enc_p.encoder.ffn_layers", "enc_p.proj"]

model = VitsModel(num_symbols=CFG["num_symbols"], num_speakers=1,
                  sample_rate=22050, batch_size=BATCH, learning_rate=LR,
                  mos_metric="none")   # UTMOS download loops + breaks rich on Colab
# load the base checkpoint into the PLAIN model, THEN wrap the convs
miss, unexp = model.load_state_dict(torch.load("/content/aegis-female-init.ckpt",
                                    map_location="cpu")["state_dict"], strict=False)
print(f"warmstart: {len(miss)} missing, {len(unexp)} unexpected")

n = inject_lora(model.model_g, TARGETS, RANK, ALPHA)
freeze_base(model.model_g)
trn = sum(p.numel() for p in lora_parameters(model.model_g))
tot = sum(p.numel() for p in model.model_g.parameters())
print(f"LoRA: wrapped {n} convs | trainable {trn} ({100*trn/tot:.2f}% of G)")

_orig = model.configure_optimizers
def _co():
    opts, scheds = _orig(); opts, scheds = list(opts), list(scheds)
    g = torch.optim.AdamW(lora_parameters(model.model_g), lr=model.hparams.learning_rate,
                          betas=model.hparams.betas, eps=model.hparams.eps)
    opts[0] = g
    scheds[0] = torch.optim.lr_scheduler.ExponentialLR(g, gamma=model.hparams.lr_decay)
    return opts, scheds
model.configure_optimizers = _co

dm = VitsDataModule(csv_path=f"{DATASET_DIR}/metadata.csv",
    audio_dir=f"{DATASET_DIR}", cache_dir="/content/cache",
    espeak_voice="ur", config_path="/content/out.onnx.json",
    voice_name="ur_PK-aegis_female", sample_rate=22050,
    num_symbols=CFG["num_symbols"], batch_size=BATCH)

trainer = L.Trainer(accelerator="gpu", devices=1, precision="16-mixed",
    max_steps=MAX_STEPS, default_root_dir="/content/train",
    log_every_n_steps=25, num_sanity_val_steps=0, enable_model_summary=False,
    callbacks=[TQDMProgressBar(refresh_rate=10),   # NOT rich (recursion-crashes on Colab)
               ModelCheckpoint(dirpath="/content/train/ckpts", save_last=True,
                               monitor="val_mel", mode="min", save_top_k=1)])
trainer.fit(model, dm)
print("training done")


## 8 · A/B — plain Urdu must be unchanged, loanwords should improve

In [ ]:
import numpy as np
from piper import PiperVoice
from IPython.display import Audio, display, Markdown

PLAIN = ["آج میں آپ کی کیا مدد کر سکتی ہوں؟", "آپ کا شکریہ، خدا حافظ۔"]
LOAN  = ["اپنے فون کے کی پیڈ پر چار ہندسے دبائیں۔",
         "آپ کے کارڈ پر ایک فراڈ ٹرانزیکشن ہے۔",
         "آپ کے اکاؤنٹ کا بیلنس پچاس ہزار روپے ہے۔"]
pid = CFG["phoneme_id_map"]
pv = PiperVoice.load("/content/aegis.onnx")
phon = {t: pv.phonemize(t) for t in PLAIN + LOAN}

def synth(scale):
    set_lora_scale(model.model_g, scale)
    model.model_g.eval()
    out = {}
    for t in PLAIN + LOAN:
        ids = [1]
        for p in (x for s in phon[t] for x in s):
            if p in pid: ids += pid[p] + [0]
        ids += [2]
        with torch.no_grad():
            a = model.model_g.infer(torch.LongTensor([ids]).to(model.device),
                torch.LongTensor([len(ids)]).to(model.device),
                noise_scale=0.667, length_scale=1.0, noise_scale_w=0.8)[0][0,0].cpu().numpy()
        out[t] = a
    return out

base, lora = synth(0.0), synth(1.0)
for t in PLAIN + LOAN:
    n = min(len(base[t]), len(lora[t]))
    d = float(np.abs(base[t][:n] - lora[t][:n]).mean())
    tag = "PLAIN" if t in PLAIN else "LOAN"
    display(Markdown(f"**{tag}** L1(base,lora)={d:.4f} — `{t}`"))
    display(Markdown("base:")); display(Audio(base[t], rate=22050))
    display(Markdown("lora:")); display(Audio(lora[t], rate=22050))
print("PLAIN L1 should be ~0 (adapter zero on those paths). LOAN L1 > 0 = it moved.")


## 9 · Merge the adapter into the weights → Piper ONNX

In [ ]:
SCALE = 1.0     # lower to 0.5 if the adapter over-corrects
set_lora_scale(model.model_g, SCALE)
merge_lora(model.model_g)          # model_g is a plain VITS again

clean = {k: v for k, v in model.hparams.items()}
torch.save({"state_dict": model.state_dict(), "hyper_parameters": clean,
            "pytorch-lightning_version": "2.0.0"}, "/content/merged.ckpt")
!sed -i 's/torch.onnx.export(/torch.onnx.export(dynamo=False,/' /content/piper1-gpl/src/piper/train/export_onnx.py 2>/dev/null; true
!cd /content/piper1-gpl && python -m piper.train.export_onnx --checkpoint /content/merged.ckpt --output-file /content/ur_PK-aegis_female-medium.onnx
import os; assert os.path.exists('/content/ur_PK-aegis_female-medium.onnx'), 'ONNX export failed — see error above'
import shutil, json
shutil.copy("/content/aegis.onnx.json", "/content/ur_PK-aegis_female-medium.onnx.json")
a = json.load(open("/content/ur_PK-aegis_female-medium.onnx.json"))["phoneme_id_map"]
b = json.load(open("/content/aegis.onnx.json"))["phoneme_id_map"]
print("phoneme_id_map:", "OK — drop-in" if a == b else "DIVERGED — do not ship")


## 10 · Final listen (the exported ONNX) + download

In [ ]:
from piper import PiperVoice
import numpy as np
from IPython.display import Audio, display, Markdown
v2 = PiperVoice.load("/content/ur_PK-aegis_female-medium.onnx")
for t in ["اپنے فون کے کی پیڈ پر چار ہندسے دبائیں۔",
          "آپ کے کارڈ پر ایک فراڈ ٹرانزیکشن ہے۔",
          "آپ کے اکاؤنٹ کا بیلنس پچاس ہزار روپے ہے۔",
          "آج میں آپ کی کیا مدد کر سکتی ہوں؟"]:
    ch = [np.frombuffer(x.audio_int16_bytes, dtype=np.int16) for x in v2.synthesize(t)]
    display(Markdown(f"`{t}`")); display(Audio(np.concatenate(ch), rate=22050))


In [ ]:
from google.colab import files
files.download("/content/ur_PK-aegis_female-medium.onnx")
files.download("/content/ur_PK-aegis_female-medium.onnx.json")
